# SigExt — New Sample Inference (no retraining)
Runs the full SigExt pipeline on a new set of samples using an already-trained Longformer checkpoint — no retraining needed.

Pipeline:
1. `prepare_data.py` — sample new examples from the dataset (different seed)
2. `inference_longformer_extractor.py` — run keyphrase extraction with the trained checkpoint
3. `zs_summarization.py` — generate summaries with the LLM + keyphrases

Outputs go to a separate directory so they can be used as an independent verification set.

In [1]:
#@title Requirements
!pip install transformers datasets accelerate evaluate rouge_score torch jsonlines rake_nltk pytorch_lightning -q
!pip install rapidfuzz -q
!pip install boto3 -q
!pip install mistralai --upgrade -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 109.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 951.5/951.5 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 8.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packag

In [2]:
#@title Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# @markdown Check this box to clone the repo from GitHub. Uncheck to load from Google Drive.

clone_repo = True # @param {type:"boolean"}
if clone_repo:
  !git clone https://github.com/lucianoselimaj/SigExt.git
  path = "/SigExt"

else:
  path = "/content/drive/MyDrive/DNLP-storage/SigExt"


Cloning into 'SigExt'...
remote: Enumerating objects: 151, done.
remote: Counting objects: 100% (151/151), done.
remote: Compressing objects: 100% (135/135), done.
remote: Total 151 (delta 74), reused 34 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (151/151), 5.12 MiB | 12.56 MiB/s, done.
Resolving deltas: 100% (74/74), done.


In [4]:
#@title Configuration

# path to the SigExt repo on your Drive
path = "/content/drive/MyDrive/ColabNotebooks/SigExt"  #@param{type:"string"}

# trained Longformer checkpoint (output of the main training notebook)
path_to_checkpoint = "experiments_claude/cnn_extractor_model/"  #@param{type:"string"}

# where to save the raw new samples
path_new_dataset    = "experiments_claude/cnn_dataset_new_samples/"  #@param{type:"string"}

# where to save the samples after keyphrase extraction
path_new_with_kp    = "experiments_claude/cnn_dataset_new_samples_with_keyphrase/"  #@param{type:"string"}

# where to save the final summaries
path_new_predictions = "experiments_claude/cnn_extsig_predictions_new_samples_k15/"  #@param{type:"string"}

# sampling config
dataset     = "cnn"   #@param{type:"string"}
new_seed    = 123     #@param{type:"integer"}   # use a different seed from the original run
n_samples   = 500     #@param{type:"integer"}
top_k_kp    = 15      #@param{type:"integer"}

print(f"Repo path:          {path}")
print(f"Checkpoint:         {path}/{path_to_checkpoint}")
print(f"Raw samples:        {path}/{path_new_dataset}")
print(f"With keyphrases:    {path}/{path_new_with_kp}")
print(f"Predictions:        {path}/{path_new_predictions}")
print(f"Seed:               {new_seed}")
print(f"Samples:            {n_samples}")
print(f"Top-K keyphrases:   {top_k_kp}")


Repo path:          /content/drive/MyDrive/ColabNotebooks/SigExt
Checkpoint:         /content/drive/MyDrive/ColabNotebooks/SigExt/experiments_claude/cnn_extractor_model/
Raw samples:        /content/drive/MyDrive/ColabNotebooks/SigExt/experiments_claude/cnn_dataset_new_samples/
With keyphrases:    /content/drive/MyDrive/ColabNotebooks/SigExt/experiments_claude/cnn_dataset_new_samples_with_keyphrase/
Predictions:        /content/drive/MyDrive/ColabNotebooks/SigExt/experiments_claude/cnn_extsig_predictions_new_samples_k15/
Seed:               123
Samples:            500
Top-K keyphrases:   15


In [5]:
#@title Setup
import os, sys

os.chdir(path)
print(f"Working directory: {os.getcwd()}")

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

# make sure the checkpoint is there before running anything
ckpt_full = f"{path}/{path_to_checkpoint}"
if os.path.exists(ckpt_full):
    files = os.listdir(ckpt_full)
    print(f"\n Checkpoint found: {ckpt_full}")
    print(f"Files: {files[:5]}")
else:
    print(f"\n Checkpoint not found: {ckpt_full}")
    print("Run the main SigExt training notebook first.")


Working directory: /content/drive/.shortcut-targets-by-id/1TpffEVc0xhfd78UxnDSQbY7er61_yzCk/SigExt

 Checkpoint found: /content/drive/MyDrive/ColabNotebooks/SigExt/experiments_claude/cnn_extractor_model/
Files: ['lightning_logs']


In [6]:
# dataset to load from HuggingFace
hf_dataset_name    = "cnn_dailymail"   #@param{type:"string"}
hf_dataset_version = "3.0.0"          #@param{type:"string"}
hf_split           = "test"           #@param{type:"string"}

# field names in the raw dataset
input_field  = "article"     #@param{type:"string"}
output_field = "highlights"  #@param{type:"string"}

# original tuning set — used only for the overlap check
path_orig_dataset = "experiments_claude/cnn_dataset_with_keyphrase/"  #@param{type:"string"}

In [7]:
#@title Step 1 — Sample new examples

import os, sys, importlib, jsonlines, numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer
from functools import partial

dataset_out = f"{path}/{path_new_dataset}"
os.makedirs(dataset_out, exist_ok=True)

if os.path.exists(f"{dataset_out}test.jsonl"):
    with jsonlines.open(f"{dataset_out}test.jsonl") as f:
        existing = list(f)
    print(f"Already exists: {len(existing)} examples — skipping.")
else:
    os.chdir(path)
    if f"{path}/src" not in sys.path:
        sys.path.insert(0, f"{path}/src")
    import prepare_data
    importlib.reload(prepare_data)

    print(f"Loading {hf_dataset_name} ({hf_split} split)...")
    raw_dataset = load_dataset(hf_dataset_name, hf_dataset_version, split=hf_split)
    tokenizer   = AutoTokenizer.from_pretrained("allenai/longformer-large-4096")

    input_processor  = partial(prepare_data.general_input_processor,  field=input_field)
    output_processor = partial(prepare_data.general_output_processor, field=output_field)

    np.random.seed(new_seed)
    print(f"Sampling {n_samples} examples (seed={new_seed})...")

    processed = prepare_data.process_split(
        data             = raw_dataset,
        num_sample       = n_samples,
        input_processor  = input_processor,
        output_processor = output_processor,
        tokenizer        = tokenizer,
        max_input_len    = 6000,
        max_output_len   = 510,
    )

    with jsonlines.open(f"{dataset_out}test.jsonl", "w") as f:
        f.write_all(processed)
    print(f"Saved {len(processed)} examples to {dataset_out}test.jsonl")

# check there's no overlap with the original tuning set
new_file  = f"{dataset_out}test.jsonl"
orig_file = f"{path}/{path_orig_dataset}test.jsonl"

if os.path.exists(new_file) and os.path.exists(orig_file):
    with jsonlines.open(new_file) as f:
        new_data = list(f)
    with jsonlines.open(orig_file) as f:
        orig_data = list(f)

    new_idx  = set(d["index"] for d in new_data  if "index" in d)
    orig_idx = set(d["index"] for d in orig_data if "index" in d)
    overlap  = new_idx & orig_idx

    print(f"\nOverlap check:")
    print(f"New:      {len(new_data)}")
    print(f"Original: {len(orig_data)}")
    print(f"Overlap:  {len(overlap)}")
    if overlap:
        print(f"Try a different new_seed to avoid overlap.")
    else:
        print(f"No overlap — sets are independent.")
elif not os.path.exists(orig_file):
    print(f"\n Overlap check skipped — original tuning set not found at {orig_file}")

Loading cnn_dailymail (test split)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Sampling 500 examples (seed=123)...


100%|██████████| 500/500 [00:14<00:00, 35.08it/s]


Saved 500 examples to /content/drive/MyDrive/ColabNotebooks/SigExt/experiments_claude/cnn_dataset_new_samples/test.jsonl

Overlap check:
New:      500
Original: 500
Overlap:  0
No overlap — sets are independent.


In [8]:
import jsonlines
import os

out = f"{path}/{path_new_dataset}test.jsonl"
if os.path.exists(out):
    with jsonlines.open(out) as f:
        saved = list(f)
    print(f"Saved: {len(saved)} examples")
    print(f"Keys:  {list(saved[0].keys())}")
else:
    print("File not found")

Saved: 500 examples
Keys:  ['raw_input', 'raw_output', 'trunc_input', 'trunc_output', 'input_length_info', 'output_length_info', 'trunc_input_phrases', 'trunc_output_phrases']


In [10]:
# @title create missing validation split

import shutil, os

dataset_dir = os.path.join(path, path_new_dataset)

test_file       = os.path.join(dataset_dir, "test.jsonl")
validation_file = os.path.join(dataset_dir, "validation.jsonl")

if os.path.exists(test_file) and not os.path.exists(validation_file):
    shutil.copy2(test_file, validation_file)
    print(f"Created validation.jsonl from test.jsonl")
elif os.path.exists(validation_file):
    print(f"validation.jsonl already exists — skipping.")
else:
    print(f"test.jsonl not found at {dataset_dir} — run Step 1 first.")

Created validation.jsonl from test.jsonl


In [11]:
#@title Step 2 — Extract keyphrases with trained Longformer checkpoint

import os

os.chdir(path)

ckpt_dir = f"{path}/experiments2/cnn_extractor_model"

kp_out   = f"{path}/{path_new_with_kp}"
data_in  = f"{path}/{path_new_dataset}"
os.makedirs(kp_out, exist_ok=True)

print(f"Checkpoint: {ckpt_dir}")
print(f"Input:      {data_in}")
print(f"Output:     {kp_out}")

if os.path.exists(f"{kp_out}test.jsonl"):
    import jsonlines
    with jsonlines.open(f"{kp_out}test.jsonl") as f:
        existing = list(f)
    print(f"\n Already extracted ({len(existing)} examples) — skipping.")
else:
    !python3 src/inference_longformer_extractor.py \
        --dataset_dir    {data_in} \
        --checkpoint_dir {ckpt_dir} \
        --output_dir     {kp_out}

# quick sanity check on the output
kp_file = f"{kp_out}test.jsonl"
if os.path.exists(kp_file):
    import jsonlines
    with jsonlines.open(kp_file) as f:
        kp_data = list(f)
    print(f"\nOutput check:")
    print(f"Examples:           {len(kp_data)}")
    print(f"Has input_kw_model: {'input_kw_model' in kp_data[0]}")
else:
    print(f"Output not found: {kp_file}")


Checkpoint: /content/drive/MyDrive/ColabNotebooks/SigExt/experiments2/cnn_extractor_model
Input:      /content/drive/MyDrive/ColabNotebooks/SigExt/experiments_claude/cnn_dataset_new_samples/
Output:     /content/drive/MyDrive/ColabNotebooks/SigExt/experiments_claude/cnn_dataset_new_samples_with_keyphrase/
INFO:root:Using f/content/drive/MyDrive/ColabNotebooks/SigExt/experiments2/cnn_extractor_model/lightning_logs/version_0/checkpoints/epoch_09-step_010000-recall20_0.200.ckpt
INFO:httpx:HTTP Request: HEAD https://huggingface.co/allenai/longformer-base-4096/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/allenai/longformer-base-4096/301e6a42cb0d9976a6d6a26a079fef81c18aa895/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/allenai/longformer-base-4096/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/res

In [ ]:
#@title Step 3 — Generate summaries (Mistral API)
# Reads keyphrases from path_new_with_kp and writes summaries to path_new_predictions.

import sys
import os
from types import ModuleType
import time

def chat(prompt, client):
    chat_response = client.chat.complete(
        model="mistral-small-2603",
        messages=[{"role": "user", "content": prompt["prompt_input"]}]
    )
    return chat_response.choices[0].message.content

def predict_one_eg_mistral(prompt):
    from mistralai.client import Mistral
    client = Mistral(api_key="YdbLhqEepgXav0x8ruTJiWR20GsgWCrD")
    err = None
    time.sleep(1.1)
    for i in range(5):
        try:
            return chat(prompt, client)
        except Exception as e:
            if "429" in str(e):
                print("Rate limit hit! Sleeping for 10 seconds...")
                time.sleep(10)
                err = e
    return f"Error: {err}"

# zs_summarization uses multiprocessing.Pool — replace with a sequential version
class FakePool:
    def __init__(self, processes=None): pass
    def __enter__(self): return self
    def __exit__(self, *args): pass
    def imap(self, func, iterable):
        return map(func, iterable)

import multiprocessing
multiprocessing.Pool = FakePool

# inject bedrock_utils so zs_summarization.py can import it without failing
m = ModuleType("bedrock_utils")
m.predict_one_eg_mistral = predict_one_eg_mistral
m.predict_one_eg_claude_instant = lambda x: "Not configured"
sys.modules["bedrock_utils"] = m

src_path = f"{path}/src"
if src_path not in sys.path:
    sys.path.append(src_path)

import importlib
import zs_summarization

os.chdir(path)

sys.argv = [
    "zs_summarization.py",
    "--model_name",         "mistral",
    "--kw_strategy",        "sigext_topk",
    "--kw_model_top_k",     str(top_k_kp),
    "--dataset",            dataset,
    "--dataset_dir",        path_new_with_kp,
    "--output_dir",         path_new_predictions,
    "--inference_on_split", "test",
]

print(f"Input:  {path}/{path_new_with_kp}")
print(f"Output: {path}/{path_new_predictions}")
print()

importlib.reload(zs_summarization)
zs_summarization.main()


In [14]:
#@title Setup Bedrock

import os
import sys
import json
import time
import traceback
import logging
from types import ModuleType

import boto3.session as boto3_session
import botocore.config

AWS_ACCESS_KEY_ID     = "" #@param{type:"string"}
AWS_SECRET_ACCESS_KEY = "" #@param{type:"string"}
AWS_REGION            = "us-east-1" #@param{type:"string"}
MODEL_ID              = "us.anthropic.claude-haiku-4-5-20251001-v1:0" #@param["mistral.mistral-7b-instruct-v0:2","us.anthropic.claude-haiku-4-5-20251001-v1:0"]

os.environ["AWS_ACCESS_KEY_ID"]     = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
os.environ["AWS_DEFAULT_REGION"]    = AWS_REGION


def _is_mistral():
    return MODEL_ID.startswith("mistral.")

def _is_claude():
    return "anthropic." in MODEL_ID  # covers both "anthropic." and "us.anthropic."


def _build_body(prompt_input):
    if _is_mistral():
        return {
            "prompt":      prompt_input,  # zs_summarization already builds the full prompt
            "max_tokens":  512,
            "temperature": 1.0,
            "top_p":       0.8,
            "top_k":       10,
        }
    elif _is_claude():
        return {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens":        512,
            "messages": [{"role": "user", "content": prompt_input}],
        }
    else:
        raise ValueError(f"Unknown model family: {MODEL_ID}")


def _parse_response(response_body):
    if _is_mistral():
        return response_body["outputs"][0]["text"]
    elif _is_claude():
        return response_body["content"][0]["text"]
    raise ValueError(f"Cannot parse response for model: {MODEL_ID}")


def predict_one_eg(x):
    current_session = boto3_session.Session()
    bedrock = current_session.client(
        service_name  = "bedrock-runtime",
        region_name   = AWS_REGION,
        endpoint_url  = f"https://bedrock-runtime.{AWS_REGION}.amazonaws.com",
        config=botocore.config.Config(
            read_timeout    = 120,
            connect_timeout = 120,
            retries         = {"max_attempts": 5},
        ),
    )

    for attempt in range(10):
        try:
            response      = bedrock.invoke_model(
                modelId     = MODEL_ID,
                contentType = "application/json",
                accept      = "*/*",
                body        = json.dumps(_build_body(x["prompt_input"])),
            )
            response_body = json.loads(response.get("body").read())
            logging.info(response_body)
            return _parse_response(response_body)
        except Exception as e:
            err = str(e)
            # don't retry on configuration errors — they won't fix themselves
            if "ValidationException" in err or "ResourceNotFoundException" in err:
                print(f"Non-retriable error: {err}")
                break
            traceback.print_exc()
            time.sleep(5)
    return ""


# sequential pool — zs_summarization uses multiprocessing.Pool internally
class FakePool:
    def __init__(self, processes=None): pass
    def __enter__(self): return self
    def __exit__(self, *args): pass
    def imap(self, func, iterable):
        return map(func, iterable)

import multiprocessing
multiprocessing.Pool = FakePool

# inject bedrock_utils so zs_summarization.py finds it at import time
m = ModuleType("bedrock_utils")
m.predict_one_eg_mistral        = predict_one_eg
m.predict_one_eg_claude_instant = predict_one_eg
sys.modules["bedrock_utils"]    = m

print(f"Backend ready — {MODEL_ID} ({AWS_REGION})")
print("Run the connection test cell to verify credentials before inference.")

Backend ready — us.anthropic.claude-haiku-4-5-20251001-v1:0 (us-east-1)
Run the connection test cell to verify credentials before inference.


In [15]:
#@title Bedrock Connection Test

print(f"Testing connection...")
print(f"  Model:  {MODEL_ID}")
print(f"  Region: {AWS_REGION}")
print()

try:
    result = predict_one_eg({"prompt_input": "Say hello in one word."})

    if result and result.strip():
        print("Connection successful")
    else:
        print("Empty response — check credentials and region.")

except Exception as e:
    err = str(e)
    print(f"Connection failed: {err}")
    print()
    if "UnrecognizedClientException" in err or "InvalidClientTokenId" in err:
        print("Invalid credentials — check AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY.")
    elif "AccessDeniedException" in err:
        print("Access denied — check that your IAM user has bedrock:InvokeModel permission.")
    elif "ResourceNotFoundException" in err:
        print(f"Model not found or not activated — go to Bedrock console → Model access and enable: {MODEL_ID}")
    elif "ValidationException" in err:
        print(f"Model requires cross-region inference profile — try prefixing the model ID with 'us.': us.{MODEL_ID}")
    elif "EndpointResolutionError" in err or "Could not connect" in err:
        print(f"Cannot reach endpoint — check that {AWS_REGION} is correct for your account.")
    else:
        print("Unexpected error — check your AWS account and IAM permissions.")

Testing connection...
  Model:  us.anthropic.claude-haiku-4-5-20251001-v1:0
  Region: us-east-1

Connection successful


In [16]:
#@title Import zs_summarization
import sys
import os

src_path = f"{path}/src"
if src_path not in sys.path:
    sys.path.append(src_path)

os.chdir(path)
print(f"Working directory: {os.getcwd()}")

import zs_summarization
print("zs_summarization imported")


Working directory: /content/drive/.shortcut-targets-by-id/1TpffEVc0xhfd78UxnDSQbY7er61_yzCk/SigExt
zs_summarization imported


In [17]:
#@title Run inference

import sys
import importlib
import os

os.chdir(path)

sys.argv = [
    "zs_summarization.py",
    "--model_name",         "mistral",
    "--kw_strategy",        "sigext_topk",
    "--kw_model_top_k",     str(top_k_kp),
    "--dataset",            dataset,
    "--dataset_dir",        path_new_with_kp,
    "--output_dir",         path_new_predictions,
    "--inference_on_split", "test",
]

print(f"Input:  {path}/{path_new_with_kp}")
print(f"Output: {path}/{path_new_predictions}")
print(f"Model:  {MODEL_ID}")
print()

importlib.reload(zs_summarization)
zs_summarization.main()


Input:  /content/drive/MyDrive/ColabNotebooks/SigExt/experiments_claude/cnn_dataset_new_samples_with_keyphrase/
Output: /content/drive/MyDrive/ColabNotebooks/SigExt/experiments_claude/cnn_extsig_predictions_new_samples_k15/
Model:  us.anthropic.claude-haiku-4-5-20251001-v1:0



100%|██████████| 500/500 [21:36<00:00,  2.59s/it]


In [18]:
#@title Verify output
import json, os

pred_path    = f"{path}/{path_new_predictions}test_predictions.json"
metrics_path = f"{path}/{path_new_predictions}test_metrics.json"
dataset_path = f"{path}/{path_new_predictions}test_dataset.jsonl"

print("Output files:")
for fpath in [pred_path, metrics_path, dataset_path]:
    exists = os.path.exists(fpath)
    size   = os.path.getsize(fpath) if exists else 0
    status = f"({size:,} bytes)" if exists else "NOT FOUND"
    print(f"  {os.path.basename(fpath)}  {status}")

if os.path.exists(pred_path):
    with open(pred_path) as f:
        preds = json.load(f)
    print(f"\nPredictions: {len(preds)}")
    print(f"Sample:\n  {preds[0][:150]}")

if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        metrics = json.load(f)
    print(f"\nROUGE metrics:")
    for k, v in metrics.items():
        print(f"  {k}: {v}")

# paths to plug into the hallucination filter notebook
print("\n" + "="*65)
print("Paths for the hallucination filter notebook:")
print(f"  path_predictions    = '{path}/{path_new_predictions}test_predictions.json'")
print(f"  path_scored_dataset = '{path}/{path_new_with_kp}test.jsonl'")
print(f"  path_full_dataset   = '{path}/{path_new_predictions}test_dataset.jsonl'")


Output files:
  test_predictions.json  (259,372 bytes)
  test_metrics.json  (296 bytes)
  test_dataset.jsonl  (16,355,827 bytes)

Predictions: 500
Sample:
  # Summary

Blundering boyfriend Jake Boys from East Sussex meant to surprise his 18-year-old partner Miss Canham with tickets to see One Direction mem

ROUGE metrics:
  rouge1p: 31.8295
  rouge1r: 50.7857
  rouge1f: 37.9207
  rouge2p: 11.4247
  rouge2r: 18.2798
  rouge2f: 13.6141
  rougeLp: 19.4118
  rougeLr: 31.3396
  rougeLf: 23.2243
  rougeLsump: 27.2176
  rougeLsumr: 43.3485
  rougeLsumf: 32.3953
  gen_len: 87.696

Paths for the hallucination filter notebook:
  path_predictions    = '/content/drive/MyDrive/ColabNotebooks/SigExt/experiments_claude/cnn_extsig_predictions_new_samples_k15/test_predictions.json'
  path_scored_dataset = '/content/drive/MyDrive/ColabNotebooks/SigExt/experiments_claude/cnn_dataset_new_samples_with_keyphrase/test.jsonl'
  path_full_dataset   = '/content/drive/MyDrive/ColabNotebooks/SigExt/experiments_cl